<a href="https://colab.research.google.com/github/psvprasad2003/SAMPLE_ML_MODELS/blob/main/madhav_executed_cp_multinomial_logistic_regression_10a_build_sequence_failure_prediction_model_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10A - Multinomial Logistic Regression for Failure Prediction (Updated)

This production-ready notebook fits one multinomial logistic regression model with four mutually exclusive classes:

- **0:** No failure within 30 days
- **1:** Failure within 7 days
- **2:** Failure in 8 to 14 days
- **3:** Failure in 15 to 30 days

It reconstructs cumulative failure probabilities as `P(7D)=P(class 1)`, `P(14D)=P(class 1)+P(class 2)`, and `P(30D)=P(class 1)+P(class 2)+P(class 3)`. It uses an engine-level split, training-only scaling, training-only negative subsampling, class weighting, validation-based regularization selection, and natural validation/test prevalence.


In [ ]:
# CELL 01 - Imports and production configuration
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    log_loss,
    precision_recall_curve,
)

ROOT = Path('/raid3/e296408/All_ECFRs_working')
BRANCH_A_DIR = ROOT / 'branch_a_ecfr_only'
PANEL_FILE = BRANCH_A_DIR / 'failure_prediction_panel' / 'ecfr_failure_prediction_panel.parquet'
OUTPUT_DIR = BRANCH_A_DIR / 'failure_prediction_panel' / 'multinomial_logistic_outputs'

HORIZONS = [7, 14, 30]
CLASS_NAMES = {
    0: 'No failure within 30D',
    1: 'Failure within 7D',
    2: 'Failure in 8-14D',
    3: 'Failure in 15-30D',
}
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
NEGATIVE_TO_FAILURE_RATIO = 20
DAYS_SINCE_LAST_SENTINEL = 9999.0
C_VALUES = [0.01, 0.1, 1.0, 10.0]
SEED = 42
MAX_ITER = 1000

def banner(title):
    print('=' * 100)
    print(title)
    print('=' * 100)

if not PANEL_FILE.is_file():
    raise FileNotFoundError(f'Panel file not found: {PANEL_FILE}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

banner('MULTINOMIAL LOGISTIC REGRESSION SETUP')
print('Panel:', PANEL_FILE)
print('Outputs:', OUTPUT_DIR)
print('Regularization C values:', C_VALUES)


MULTINOMIAL LOGISTIC REGRESSION SETUP
Panel: /raid3/e296408/All_ECFRs_working/branch_a_ecfr_only/failure_prediction_panel/ecfr_failure_prediction_panel.parquet
Outputs: /raid3/e296408/All_ECFRs_working/branch_a_ecfr_only/failure_prediction_panel/multinomial_logistic_outputs
Regularization C values: [0.01, 0.1, 1.0, 10.0]


In [ ]:
# CELL 02 - Load parquet and verify expected schema
def load_and_verify_panel(panel_file):
    panel_file = Path(panel_file)
    if not panel_file.is_file():
        raise FileNotFoundError(f'Panel file not found: {panel_file}')

    panel = pd.read_parquet(panel_file)
    if panel.empty:
        raise ValueError('The parquet file is empty.')

    labels = [f'FAILS_WITHIN_{h}D' for h in HORIZONS]
    required = ['ENGINE_SERIAL', 'SNAPSHOT_TIME', 'ENGINE_EVER_FAILS_IN_RECORD'] + labels
    missing = [c for c in required if c not in panel.columns]
    if missing:
        raise ValueError(f'Schema mismatch. Missing required columns: {missing}')

    if panel['ENGINE_SERIAL'].isna().any():
        raise ValueError('ENGINE_SERIAL contains missing values.')
    panel['SNAPSHOT_TIME'] = pd.to_datetime(panel['SNAPSHOT_TIME'], errors='coerce')
    if panel['SNAPSHOT_TIME'].isna().any():
        raise ValueError('SNAPSHOT_TIME contains invalid or missing values.')

    for c in labels + ['ENGINE_EVER_FAILS_IN_RECORD']:
        vals = set(panel[c].dropna().unique().tolist())
        if panel[c].isna().any() or not vals.issubset({0, 1, False, True}):
            raise ValueError(f'{c} must be complete and binary 0/1. Found: {list(vals)[:10]}')
        panel[c] = panel[c].astype(np.int8)

    bad_nested = (panel[labels[0]] > panel[labels[1]]) | (panel[labels[1]] > panel[labels[2]])
    if bad_nested.any():
        raise ValueError(f'{int(bad_nested.sum()):,} rows violate 7D <= 14D <= 30D label nesting.')

    non_features = set(required)
    feature_cols = [c for c in panel.columns if c not in non_features]
    if not feature_cols:
        raise ValueError('No predictive feature columns were found.')
    non_numeric = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(panel[c])]
    if non_numeric:
        raise ValueError(f'Non-numeric features found: {non_numeric[:30]}')

    panel = panel.sort_values(['ENGINE_SERIAL', 'SNAPSHOT_TIME']).reset_index(drop=True)
    schema = pd.DataFrame({
        'column': panel.columns,
        'dtype': panel.dtypes.astype(str).values,
        'missing_count': panel.isna().sum().values,
        'missing_pct': (100 * panel.isna().mean()).round(4).values,
    })

    banner('SCHEMA VERIFIED')
    print(f'Rows: {len(panel):,} | Columns: {panel.shape[1]:,} | Engines: {panel.ENGINE_SERIAL.nunique():,}')
    print('Date range:', panel.SNAPSHOT_TIME.min(), 'to', panel.SNAPSHOT_TIME.max())
    print('Features:', len(feature_cols))
    print('Duplicate engine/timestamp rows:', int(panel.duplicated(['ENGINE_SERIAL', 'SNAPSHOT_TIME']).sum()))
    for c in labels:
        print(f'{c}: {int(panel[c].sum()):,} positives ({panel[c].mean():.6%})')
    display(schema.head(35))
    return panel, labels, feature_cols, schema


In [ ]:
# CELL 03 - Build exclusive multinomial target and prepare features
def prepare_data(panel, labels, feature_cols):
    panel = panel.copy()
    days_cols = [c for c in feature_cols if c.endswith('__DAYS_SINCE_LAST')]
    other_cols = [c for c in feature_cols if c not in days_cols]

    panel[other_cols] = panel[other_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if days_cols:
        panel[days_cols] = (
            panel[days_cols]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(DAYS_SINCE_LAST_SENTINEL)
        )

    X = panel[feature_cols].to_numpy(dtype=np.float32)
    if not np.isfinite(X).all():
        raise ValueError('Features contain NaN/infinite values after imputation.')

    y7, y14, y30 = [panel[c].to_numpy(dtype=np.int8) for c in labels]
    y_multi = np.select(
        [y7 == 1, (y14 == 1) & (y7 == 0), (y30 == 1) & (y14 == 0)],
        [1, 2, 3],
        default=0,
    ).astype(np.int8)
    panel['FAILURE_TIME_CLASS'] = y_multi

    banner('MULTINOMIAL TARGET')
    counts = pd.Series(y_multi).value_counts().sort_index()
    for k in range(4):
        print(f'{k}: {CLASS_NAMES[k]} = {int(counts.get(k, 0)):,}')
    missing_classes = set(range(4)) - set(np.unique(y_multi))
    if missing_classes:
        raise ValueError(f'Classes absent from full data: {sorted(missing_classes)}')

    print('Feature matrix:', X.shape)
    print('Zero-filled columns:', len(other_cols))
    print('Sentinel-filled DAYS_SINCE_LAST columns:', len(days_cols))
    return panel, X, y_multi, days_cols


In [ ]:
# CELL 04 - Engine-level fail-stratified split and training subsample
def split_and_subsample(panel, y_multi):
    ever = (
        panel.groupby('ENGINE_SERIAL')['ENGINE_EVER_FAILS_IN_RECORD']
        .first().astype(int).to_dict()
    )
    rng = np.random.RandomState(SEED)
    splits = {'train': [], 'val': [], 'test': []}
    engines = np.array(list(ever), dtype=object)
    strat = np.array([ever[e] for e in engines])

    for label in [0, 1]:
        pool = engines[strat == label].copy()
        rng.shuffle(pool)
        a = int(len(pool) * TRAIN_FRAC)
        b = a + int(len(pool) * VAL_FRAC)
        splits['train'] += list(pool[:a])
        splits['val'] += list(pool[a:b])
        splits['test'] += list(pool[b:])

    splits = {k: set(v) for k, v in splits.items()}
    assert not (splits['train'] & splits['val'])
    assert not (splits['train'] & splits['test'])
    assert not (splits['val'] & splits['test'])

    row_split = np.where(
        panel.ENGINE_SERIAL.isin(splits['train']), 'train',
        np.where(panel.ENGINE_SERIAL.isin(splits['val']), 'val', 'test')
    )
    full_train = np.where(row_split == 'train')[0]
    val_idx = np.where(row_split == 'val')[0]
    test_idx = np.where(row_split == 'test')[0]

    failure_idx = full_train[y_multi[full_train] > 0]
    no_failure_idx = full_train[y_multi[full_train] == 0]
    if len(failure_idx) == 0:
        raise ValueError('No failure-class rows in training split.')

    n0 = min(len(no_failure_idx), len(failure_idx) * NEGATIVE_TO_FAILURE_RATIO)
    keep0 = rng.choice(no_failure_idx, size=n0, replace=False)
    train_idx = np.concatenate([failure_idx, keep0])
    rng.shuffle(train_idx)

    required_classes = {0, 1, 2, 3}
    train_classes = set(np.unique(y_multi[train_idx]))
    if not required_classes.issubset(train_classes):
        raise ValueError(f'Training sample is missing classes: {sorted(required_classes - train_classes)}')

    banner('LEAKAGE-SAFE SPLIT')
    for s in ['train', 'val', 'test']:
        idx = np.where(row_split == s)[0]
        counts = dict(pd.Series(y_multi[idx]).value_counts().sort_index())
        print(f'{s:>5}: {len(splits[s]):,} engines | {len(idx):,} natural rows | classes {counts}')
    print('Subsampled training rows:', len(train_idx))
    print('Subsampled class counts:', dict(pd.Series(y_multi[train_idx]).value_counts().sort_index()))
    return splits, row_split, train_idx, val_idx, test_idx


In [ ]:
# CELL 05 - Scaling, training, tuning, and probability utilities
def aligned_probabilities(model, X):
    raw = model.predict_proba(X)
    out = np.zeros((len(X), 4), dtype=np.float64)
    for j, c in enumerate(model.classes_):
        out[:, int(c)] = raw[:, j]
    return out

def horizon_probabilities(p):
    return {
        7: p[:, 1],
        14: p[:, 1] + p[:, 2],
        30: p[:, 1] + p[:, 2] + p[:, 3],
    }

def binary_metric(y, p):
    if np.unique(y).size < 2:
        return {'roc_auc': np.nan, 'pr_auc': np.nan}
    return {
        'roc_auc': roc_auc_score(y, p),
        'pr_auc': average_precision_score(y, p),
    }

def horizon_truth(y, horizon):
    if horizon == 7:
        return (y == 1).astype(int)
    if horizon == 14:
        return np.isin(y, [1, 2]).astype(int)
    return (y > 0).astype(int)

def fit_and_tune(X, y, train_idx, val_idx):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X[train_idx])
    Xv = scaler.transform(X[val_idx])

    tuning = []
    best_model = None
    best_C = None
    best_score = -np.inf

    for C in C_VALUES:
        model = LogisticRegression(
            C=C,
            solver='lbfgs',
            max_iter=MAX_ITER,
            class_weight='balanced',
            random_state=SEED,
        )
        model.fit(Xtr, y[train_idx])
        pv = aligned_probabilities(model, Xv)
        hp = horizon_probabilities(pv)
        row = {
            'C': C,
            'val_multiclass_log_loss': log_loss(y[val_idx], pv, labels=[0, 1, 2, 3]),
            'n_iter_max': int(np.max(model.n_iter_)),
        }
        scores = []
        for h in HORIZONS:
            truth = horizon_truth(y[val_idx], h)
            m = binary_metric(truth, hp[h])
            row[f'val_roc_auc_{h}d'] = m['roc_auc']
            row[f'val_pr_auc_{h}d'] = m['pr_auc']
            if not np.isnan(m['pr_auc']):
                scores.append(m['pr_auc'])
        row['val_mean_pr_auc'] = float(np.mean(scores)) if scores else np.nan
        tuning.append(row)

        score = -np.inf if np.isnan(row['val_mean_pr_auc']) else row['val_mean_pr_auc']
        if score > best_score:
            best_score = score
            best_model = model
            best_C = C

    if best_model is None:
        raise RuntimeError('Model selection failed because validation metrics were unavailable.')
    return scaler, best_model, best_C, pd.DataFrame(tuning)


In [ ]:
# CELL 06 - Evaluation and artifact saving
def run_pipeline(panel_file=PANEL_FILE, output_dir=OUTPUT_DIR):
    started = time.time()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    panel, labels, features, schema = load_and_verify_panel(panel_file)
    panel, X, y, days_cols = prepare_data(panel, labels, features)
    splits, row_split, tr, va, te = split_and_subsample(panel, y)
    scaler, model, best_C, tuning = fit_and_tune(X, y, tr, va)

    X_val_scaled = scaler.transform(X[va])
    X_test_scaled = scaler.transform(X[te])
    p_val = aligned_probabilities(model, X_val_scaled)
    p_test = aligned_probabilities(model, X_test_scaled)
    hv = horizon_probabilities(p_val)
    ht = horizon_probabilities(p_test)

    performance = []
    thresholds = []
    predictions = pd.DataFrame({
        'ENGINE_SERIAL': panel.loc[te, 'ENGINE_SERIAL'].values,
        'SNAPSHOT_TIME': panel.loc[te, 'SNAPSHOT_TIME'].values,
        'ACTUAL_CLASS': y[te],
        'PREDICTED_CLASS': model.predict(X_test_scaled),
    })
    for k in range(4):
        predictions[f'PROB_CLASS_{k}'] = p_test[:, k]

    banner('HELD-OUT TEST PERFORMANCE')
    for h in HORIZONS:
        yv = horizon_truth(y[va], h)
        yt = horizon_truth(y[te], h)
        mv = binary_metric(yv, hv[h])
        mt = binary_metric(yt, ht[h])

        precision, recall, thr = precision_recall_curve(yv, hv[h])
        candidates = np.where(recall[:-1] >= 0.80)[0]
        threshold = (
            float(thr[candidates[np.argmax(precision[:-1][candidates])]])
            if len(candidates) else 0.5
        )
        cm05 = confusion_matrix(yt, (ht[h] >= 0.5).astype(int), labels=[0, 1])
        cmt = confusion_matrix(yt, (ht[h] >= threshold).astype(int), labels=[0, 1])

        performance.append({
            'HORIZON_DAYS': h,
            'VAL_ROC_AUC': mv['roc_auc'],
            'VAL_PR_AUC': mv['pr_auc'],
            'TEST_ROC_AUC': mt['roc_auc'],
            'TEST_PR_AUC': mt['pr_auc'],
        })
        thresholds.append({
            'HORIZON_DAYS': h,
            'VALIDATION_THRESHOLD_80_RECALL': threshold,
            'TEST_CM_0.5': cm05.tolist(),
            'TEST_CM_TUNED': cmt.tolist(),
        })
        predictions[f'LABEL_{h}D'] = yt
        predictions[f'PRED_PROB_{h}D'] = ht[h]
        predictions[f'PRED_CLASS_{h}D_TUNED'] = (ht[h] >= threshold).astype(int)
        print(
            f'{h}D: Test ROC-AUC={mt["roc_auc"]:.4f}, '
            f'PR-AUC={mt["pr_auc"]:.4f}, threshold={threshold:.6f}'
        )

    multiclass_test_log_loss = log_loss(y[te], p_test, labels=[0, 1, 2, 3])
    print('Best C:', best_C)
    print('Multiclass test log loss:', multiclass_test_log_loss)
    print(classification_report(
        y[te], predictions.PREDICTED_CLASS,
        labels=[0, 1, 2, 3],
        target_names=[CLASS_NAMES[i] for i in range(4)],
        zero_division=0,
    ))

    performance = pd.DataFrame(performance)
    thresholds = pd.DataFrame(thresholds)
    display(performance)

    coefficients = []
    for row_idx, class_id in enumerate(model.classes_):
        for feature, value in zip(features, model.coef_[row_idx]):
            coefficients.append({
                'class': int(class_id),
                'class_name': CLASS_NAMES[int(class_id)],
                'feature': feature,
                'coefficient': value,
                'abs_coefficient': abs(value),
            })
    coefficients = pd.DataFrame(coefficients).sort_values(
        ['class', 'abs_coefficient'], ascending=[True, False]
    )

    joblib.dump({
        'model': model,
        'scaler': scaler,
        'feature_cols': features,
        'class_names': CLASS_NAMES,
    }, output_dir / 'multinomial_logistic_model.joblib')
    schema.to_csv(output_dir / 'verified_schema.csv', index=False)
    tuning.to_csv(output_dir / 'regularization_tuning.csv', index=False)
    performance.to_csv(output_dir / 'test_performance_summary.csv', index=False)
    thresholds.to_csv(output_dir / 'threshold_sensitivity.csv', index=False)
    predictions.to_csv(output_dir / 'test_predictions.csv', index=False)
    coefficients.to_csv(output_dir / 'model_coefficients.csv', index=False)

    config = {
        'panel_file': str(panel_file),
        'output_dir': str(output_dir),
        'feature_cols': features,
        'days_since_cols': days_cols,
        'class_names': CLASS_NAMES,
        'best_C': best_C,
        'seed': SEED,
        'negative_to_failure_ratio': NEGATIVE_TO_FAILURE_RATIO,
        'multiclass_test_log_loss': multiclass_test_log_loss,
    }
    (output_dir / 'model_config.json').write_text(json.dumps(config, indent=2, default=str))

    banner('PIPELINE COMPLETE')
    print('Artifacts:', output_dir)
    for p in sorted(output_dir.iterdir()):
        print(' -', p.name)
    print(f'Elapsed: {time.time() - started:.1f}s')

    return {
        'model': model,
        'scaler': scaler,
        'performance': performance,
        'thresholds': thresholds,
        'predictions': predictions,
        'coefficients': coefficients,
        'tuning': tuning,
        'output_dir': output_dir,
        'splits': splits,
    }


## Run the complete pipeline

The input and output paths are defined in Cell 01. Run all cells in order.


In [ ]:
# CELL 07 - Run
results = run_pipeline(PANEL_FILE, OUTPUT_DIR)


SCHEMA VERIFIED
Rows: 1,277,405 | Columns: 35 | Engines: 1,621
Date range: 1969-02-09 20:16:54 to 2068-05-05 16:57:12
Features: 29
Duplicate engine/timestamp rows: 0
FAILS_WITHIN_7D: 188 positives (0.014717%)
FAILS_WITHIN_14D: 334 positives (0.026147%)
FAILS_WITHIN_30D: 609 positives (0.047675%)


,column,dtype,missing_count,missing_pct
0,ENGINE_SERIAL,object,0,0.0000
1,SNAPSHOT_TIME,datetime64[ns],0,0.0000
2,chip__COUNT_LOOKBACK,int64,0,0.0000
3,chip__RATE_PER_DAY,float64,0,0.0000
4,chip__DAYS_SINCE_LAST,float64,1238841,96.9811
5,clm_performance__COUNT_LOOKBACK,int64,0,0.0000
6,clm_performance__RATE_PER_DAY,float64,0,0.0000
7,clm_performance__DAYS_SINCE_LAST,float64,744325,58.2685
8,cru_performance__COUNT_LOOKBACK,int64,0,0.0000
9,cru_performance__RATE_PER_DAY,float64,0,0.0000


MULTINOMIAL TARGET
0: No failure within 30D = 1,276,796
1: Failure within 7D = 188
2: Failure in 8-14D = 146
3: Failure in 15-30D = 275
Feature matrix: (1277405, 29)
Zero-filled columns: 20
Sentinel-filled DAYS_SINCE_LAST columns: 9
LEAKAGE-SAFE SPLIT
train: 1,134 engines | 906,113 natural rows | classes {0: np.int64(905672), 1: np.int64(131), 2: np.int64(106), 3: np.int64(204)}
  val: 242 engines | 185,851 natural rows | classes {0: np.int64(185774), 1: np.int64(28), 2: np.int64(18), 3: np.int64(31)}
 test: 245 engines | 185,441 natural rows | classes {0: np.int64(185350), 1: np.int64(29), 2: np.int64(22), 3: np.int64(40)}
Subsampled training rows: 9261
Subsampled class counts: {0: np.int64(8820), 1: np.int64(131), 2: np.int64(106), 3: np.int64(204)}
HELD-OUT TEST PERFORMANCE
7D: Test ROC-AUC=0.8660, PR-AUC=0.0105, threshold=0.223046
14D: Test ROC-AUC=0.8586, PR-AUC=0.0066, threshold=0.475954
30D: Test ROC-AUC=0.8205, PR-AUC=0.0171, threshold=0.497253
Best C: 0.01
Multiclass test log 

,HORIZON_DAYS,VAL_ROC_AUC,VAL_PR_AUC,TEST_ROC_AUC,TEST_PR_AUC
0,7,0.905462,0.018351,0.866015,0.010483
1,14,0.876219,0.010754,0.858585,0.006576
2,30,0.777359,0.015897,0.820461,0.017085


PIPELINE COMPLETE
Artifacts: /raid3/e296408/All_ECFRs_working/branch_a_ecfr_only/failure_prediction_panel/multinomial_logistic_outputs
 - model_coefficients.csv
 - model_config.json
 - multinomial_logistic_model.joblib
 - regularization_tuning.csv
 - test_performance_summary.csv
 - test_predictions.csv
 - threshold_sensitivity.csv
 - verified_schema.csv
Elapsed: 3.1s
